# TRINETRA: train the UATD detector (Colab)

**Before you start**
1. Menu **Runtime → Change runtime type → T4 GPU**.
2. Download `UATD_Training.zip` from the UATD Figshare page (search *UATD dataset figshare*) and upload it to the **root of your Google Drive** (My Drive).
3. Run every cell from top to bottom (Runtime → Run all). Training takes roughly 1 to 3 hours on a T4.

At the end you download `uatd_yolov8.pt` and put it in `backend/weights/`.

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
assert torch.cuda.is_available(), 'Switch the runtime to GPU first'

In [ ]:
!pip -q install ultralytics opencv-python-headless

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!ls -lh /content/drive/MyDrive/UATD_Training.zip
!unzip -q -o /content/drive/MyDrive/UATD_Training.zip -d /content/uatd_raw
!find /content/uatd_raw -maxdepth 2 | head -20

In [ ]:
%%writefile prepare_uatd.py
"""Convert UATD (BMP images + XML annotations) into YOLO format.

Usage:  python prepare_uatd.py <path-to-unzipped-UATD_Training> <output_dir>
Classes are discovered from the XML files themselves, so no names are hard-coded.
Assumes VOC-style <object><name/><bndbox><xmin/><ymin/><xmax/><ymax/></bndbox></object>;
if your XML differs, the script prints a sample so the parser can be adjusted.
"""
import random
import shutil
import sys
import xml.etree.ElementTree as ET
from pathlib import Path

import cv2


def parse(xml_path):
    objs = []
    for o in ET.parse(xml_path).getroot().iter("object"):
        name = (o.findtext("name") or "").strip()
        bb = o.find("bndbox")
        if not name or bb is None:
            continue
        objs.append((name, *[float(bb.findtext(k)) for k in ("xmin", "ymin", "xmax", "ymax")]))
    return objs


def main(src, out, val_frac=0.1, seed=0):
    src, out = Path(src), Path(out)
    xmls = {p.stem: p for p in src.rglob("*.xml")}
    imgs = [p for p in sorted(src.rglob("*.bmp")) if p.stem in xmls]
    if not imgs:
        sys.exit("No matching .bmp/.xml pairs found. Check the path.")

    parsed = {p.stem: parse(xmls[p.stem]) for p in imgs}
    classes = sorted({o[0] for objs in parsed.values() for o in objs})
    if not classes:
        print(xmls[imgs[0].stem].read_text()[:1500])
        sys.exit("Could not read objects from the XML (sample above). Adjust parse().")
    cid = {c: i for i, c in enumerate(classes)}

    random.Random(seed).shuffle(imgs)
    n_val = max(1, int(len(imgs) * val_frac))
    for i, img in enumerate(imgs):
        split = "val" if i < n_val else "train"
        (out / "images" / split).mkdir(parents=True, exist_ok=True)
        (out / "labels" / split).mkdir(parents=True, exist_ok=True)
        h, w = cv2.imread(str(img), cv2.IMREAD_GRAYSCALE).shape
        shutil.copy(img, out / "images" / split / img.name)
        lines = []
        for name, x1, y1, x2, y2 in parsed[img.stem]:
            cx, cy = (x1 + x2) / 2 / w, (y1 + y2) / 2 / h
            lines.append(f"{cid[name]} {cx:.6f} {cy:.6f} {(x2 - x1) / w:.6f} {(y2 - y1) / h:.6f}")
        (out / "labels" / split / f"{img.stem}.txt").write_text("\n".join(lines))

    yaml = f"path: {out.resolve()}\ntrain: images/train\nval: images/val\nnames:\n"
    yaml += "".join(f"  {i}: {c}\n" for i, c in enumerate(classes))
    (out / "uatd.yaml").write_text(yaml)
    print(f"{len(imgs)} images, classes: {classes}\nWrote {out / 'uatd.yaml'}")


if __name__ == "__main__":
    if len(sys.argv) < 3:
        sys.exit(__doc__)
    main(sys.argv[1], sys.argv[2])


In [ ]:
%%writefile train.py
"""Train YOLOv8 on the converted UATD data and install the result for the API.

Usage (from backend/):  python training/train.py <path-to>/uatd.yaml [epochs]
Needs internet the first time (downloads yolov8s.pt). A GPU is strongly recommended.
"""
import shutil
import sys
from pathlib import Path

from ultralytics import YOLO

data = sys.argv[1]
epochs = int(sys.argv[2]) if len(sys.argv) > 2 else 100

model = YOLO("yolov8s.pt")
model.train(data=data, imgsz=640, epochs=epochs, batch=16, project="runs", name="uatd",
            hsv_h=0.0, hsv_s=0.0,  # sonar is grayscale: no colour jitter
            patience=25)
Path("weights").mkdir(exist_ok=True)
shutil.copy(model.trainer.best, "weights/uatd_yolov8.pt")
print("Installed weights/uatd_yolov8.pt: restart the API and demo mode switches off.")


In [ ]:
# Convert to YOLO format. Check that the class list printed at the end looks right (10 classes).
!python prepare_uatd.py /content/uatd_raw /content/data/uatd

In [ ]:
# Quick 2-epoch smoke test first. If this works, run the full training in the next cell.
!python train.py /content/data/uatd/uatd.yaml 2

In [ ]:
# Full training (edit 100 if you want fewer/more epochs)
!python train.py /content/data/uatd/uatd.yaml 100

In [ ]:
# Accuracy per class. Look at mAP50: higher is better.
from ultralytics import YOLO
model = YOLO('weights/uatd_yolov8.pt')
m = model.val(data='/content/data/uatd/uatd.yaml')
print('mAP50:', round(m.box.map50, 3))
for i, name in model.names.items():
    print(f'{name:15s} mAP50 = {m.box.maps[i]:.3f}')

In [ ]:
# See it name objects on real validation images
import glob
from IPython.display import Image, display
imgs = sorted(glob.glob('/content/data/uatd/images/val/*'))[:4]
res = model.predict(imgs, conf=0.25, save=True, project='preview', name='val', exist_ok=True)
for r in res:
    print([(model.names[int(b.cls)], round(float(b.conf), 2)) for b in r.boxes])
for p in sorted(glob.glob('preview/val/*')): display(Image(p, width=500))

In [ ]:
# Save a copy to Drive and download it
!cp weights/uatd_yolov8.pt /content/drive/MyDrive/uatd_yolov8.pt
from google.colab import files
files.download('weights/uatd_yolov8.pt')

**Next:** put `uatd_yolov8.pt` in `backend/weights/`, restart the backend, and open `http://localhost:8000/health`. It should show `"demo_mode": false`.